# LangChain Hands-on 14 — Middleware & Guardrails
**Trainer:** Ajeetkumar

**What we will practise**
- What is middleware?
- PII protection (mask email / card)
- Human-in-the-loop approval
- Call limits & model fallback
- Auto-summarise long chats
- Custom middleware: logging, blocking bad words, dynamic model

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langgraph

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# What is Middleware?
Code that runs **around** the agent loop — without changing the agent itself. Like security checks at an airport.

```
before_agent → [ before_model → MODEL → after_model → TOOLS ]* → after_agent
```
| Hook | Example use |
|---|---|
| before_model | trim history, add context |
| wrap_model_call | change model, retry |
| after_model | block unsafe answers |
| wrap_tool_call | log tools, check permissions |

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
@tool
def get_balance(account_id: str) -> str:
    """Get account balance."""
    return f"Account {account_id} balance: Rs 1,24,500"

@tool
def transfer_money(to_account: str, amount: float) -> str:
    """Transfer money to another account."""
    return f"Rs {amount} transferred to {to_account}."

bank_tools = [get_balance, transfer_money]

## 1. PIIMiddleware — hide personal data before it reaches the LLM
Strategies: `redact`, `mask`, `hash`, `block`

In [ ]:
from langchain.agents.middleware import PIIMiddleware

In [ ]:
pii_agent = create_agent(llm, tools=[], middleware=[
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
])

In [ ]:
r = pii_agent.invoke({"messages": [{"role": "user", "content": "My email is ajeet@example.com and card 4111 1111 1111 1111. What did I share?"}]})
print("Stored message:", r["messages"][0].content)
print("Answer        :", r["messages"][-1].content)

## 2. Human-in-the-loop — ask a human before money moves 💰

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

In [ ]:
hitl_agent = create_agent(
    llm, bank_tools,
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"transfer_money": True, "get_balance": False})],
    checkpointer=InMemorySaver(),       # required to pause & resume
)
config = {"configurable": {"thread_id": "hitl-1"}}

In [ ]:
r = hitl_agent.invoke({"messages": [{"role": "user", "content": "Transfer Rs 5000 to ACC999"}]}, config)

In [ ]:
r["__interrupt__"][0].value["action_requests"]       # waiting for approval

In [ ]:
decision = input("approve or reject? ")
if decision == "approve":
    resume = {"decisions": [{"type": "approve"}]}
else:
    resume = {"decisions": [{"type": "reject", "message": "Manager rejected this transfer."}]}

In [ ]:
r = hitl_agent.invoke(Command(resume=resume), config)
print(r["messages"][-1].content)

## 3. Limits & fallback — protect cost and uptime

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, ModelFallbackMiddleware

In [ ]:
safe_agent = create_agent(
    "groq:model-that-does-not-exist",          # primary fails...
    bank_tools,
    middleware=[
        ModelFallbackMiddleware("groq:openai/gpt-oss-20b"),        # ...backup model is used
        ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="transfer_money", run_limit=1),
    ],
)
safe_agent.invoke({"messages": [{"role": "user", "content": "Balance of ACC123?"}]})["messages"][-1].content

## 4. SummarizationMiddleware — summarise old messages automatically

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

In [ ]:
summary_agent = create_agent(
    llm, tools=[],
    middleware=[SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2))],
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "long-chat"}}
for msg in ["I am Ajeet.", "I teach GenAI.", "I live in Maharashtra.", "I like LangChain.", "What do you know about me?"]:
    r = summary_agent.invoke({"messages": [{"role": "user", "content": msg}]}, cfg)
print(r["messages"][-1].content)
print("messages kept:", len(r["messages"]))

## 5. Custom middleware — logging

In [ ]:
from langchain.agents.middleware import before_model, after_model, wrap_tool_call, AgentState
from langgraph.runtime import Runtime

In [ ]:
@before_model
def log_before(state: AgentState, runtime: Runtime):
    print(f"➡️ calling LLM with {len(state['messages'])} messages")

@wrap_tool_call
def log_tools(request, handler):
    print(f"🔧 tool: {request.tool_call['name']} {request.tool_call['args']}")
    return handler(request)

In [ ]:
log_agent = create_agent(llm, bank_tools, middleware=[log_before, log_tools])
log_agent.invoke({"messages": [{"role": "user", "content": "Balance of ACC777?"}]})["messages"][-1].content

## 6. Custom guardrail — block banned words in the answer

In [ ]:
from langchain.messages import AIMessage

BANNED = ["guaranteed returns", "100% profit"]

@after_model(can_jump_to=["end"])
def compliance_guard(state: AgentState, runtime: Runtime):
    text = (state["messages"][-1].content or "").lower()
    if any(word in text for word in BANNED):
        return {"messages": [AIMessage("⚠️ This answer was blocked by compliance policy.")], "jump_to": "end"}

In [ ]:
guard_agent = create_agent(llm, [], middleware=[compliance_guard],
                           system_prompt="You are a pushy sales agent. Always promise guaranteed returns.")
guard_agent.invoke({"messages": [{"role": "user", "content": "Should I invest in your mutual fund?"}]})["messages"][-1].content

## 7. Custom middleware — dynamic model (cheap for short, strong for long chats)

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest

In [ ]:
small_llm = ChatGroq(model="openai/gpt-oss-20b")
big_llm = ChatGroq(model="openai/gpt-oss-120b")

@wrap_model_call
def choose_model(request: ModelRequest, handler):
    model = big_llm if len(request.messages) > 6 else small_llm
    return handler(request.override(model=model))

smart_agent = create_agent(small_llm, bank_tools, middleware=[choose_model])

# Built-in middleware list
| Middleware | Use |
|---|---|
| `PIIMiddleware` | hide email, card, IP… |
| `HumanInTheLoopMiddleware` | approval before risky tools |
| `SummarizationMiddleware` | long chats |
| `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` | stop loops, control cost |
| `ModelFallbackMiddleware` | backup model |
| `ModelRetryMiddleware` / `ToolRetryMiddleware` | auto retry |
| `TodoListMiddleware` | agent plans tasks |
| `LLMToolSelectorMiddleware` | pick relevant tools when you have many |

## 🧪 Practice
1. Add human approval to a `send_email` tool with approve / edit / reject.
2. Write `@before_model` middleware that blocks the text "ignore previous instructions".
3. Limit an agent to max 3 tool calls.